In [ ]:
from datasets_collector.dataset import DatasetManager
url = "https://github.com/karpathy/char-rnn/blob/master/data/tinyshakespeare/input.txt"

dataset_manager = DatasetManager(url)
dataset_stats = dataset_manager.statistics
print("Dataset statistics")
print(f"Characters:        {dataset_stats['characters']:,}")
print(f"UTF-8 bytes:       {dataset_stats['utf8_bytes']:,}")
print(f"Lines:             {dataset_stats['lines']:,}")
print(f"Words:             {dataset_stats['words']:,}")
print(f"Unique words:      {dataset_stats['unique_words']:,}")
print("\nMost common words:")

for word, count in dataset_stats["most_common_words"]:
    print(f"  {word:<16} {count:>6,}")

print("\nTraining set preview:")
print(dataset_manager.train_set[:100])  # Print the first 100 characters of the training set
print("\nValidation set preview:")
print(dataset_manager.val_set[:100])  # Print the first 100 characters of the validation set

In [ ]:
# BPE Tokenizer from trained from input text
from tokenization.tokenizer import BPETokenizer
train_input_text = dataset_manager.train_set  # Use the training set for tokenizer training
validation_input_text = dataset_manager.val_set  # Use the validation set for tokenizer validation

# Initialize the tokenizer
my_tokenizer = BPETokenizer()

vocabulary_size = 300  # Set the desired vocabulary size for the tokenizer
# ====== Train the tokenizer ===== 
# on the input text with a vocabulary size of 300 tokens
my_tokenizer.train(train_input_text, vocab_size=vocabulary_size)
my_tokenizer.save_tokenizer('my_BPE_tinyshakespeare_tokenizer.json')

# ====== Test the tokenizer ======
# Encode the input text into tokens and then decode it back to text
train_tokens = my_tokenizer.encode(train_input_text)
train_decoded_text = my_tokenizer.decode(train_tokens)
assert train_decoded_text == train_input_text, "Decoded training text does not match the original input text."
print(f"Training text (decoded): {train_decoded_text[:100]}")

# ===== Validate encode and decode =====
validation_tokens = my_tokenizer.encode(validation_input_text)
validation_decoded_text = my_tokenizer.decode(validation_tokens)
assert validation_decoded_text == validation_input_text, "Decoded validation text does not match the original input text."
print(f"Validation text (decoded): {validation_decoded_text[:100]}")  # Print the first 100 characters of the decoded text

print(f"Vocabulary size: {len(my_tokenizer.vocab)}")

In [ ]:
import torch

# number of independent sequences to process in parallel
batch_size = 4
# maximum number of characters to consider in the past for predicting the next character
context_length = 16

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_tokens if split == 'train' else validation_tokens
    # select random starting indices for the batch
    ix = torch.randint(len(data) - context_length, (batch_size,))
    # create input and target sequences based on the random indices
    x = torch.stack([data[i:i+context_length] for i in ix])
    y = torch.stack([data[i+1:i+context_length+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)


In [ ]:
from models.bigram_language_model import BigramLanguageModel

vocab_size = len(my_tokenizer.vocab)
model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print(logits.shape)
print(loss)

# Generate some text from the model starting from the initial token with index 0 
# generating a maximum of 100 new tokens
print(my_tokenizer.decode(model.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0]))


In [ ]:
from trainer.trainer import Trainer

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
trainer = Trainer(model, optimizer)

# Train the model for a specified number of iterations
num_iterations = 100000

for iteration in range(num_iterations):
    # Get a batch of training data
    xb, yb = get_batch('train')
    
    # Perform a training step and get the loss
    loss = trainer.train_step(xb, yb)
    
    # Print the loss every 100 iterations
    if iteration % 10000 == 0:
        print(f"Iteration {iteration}, Loss: {loss:.4f}")
        # Generate some text from the model starting from the initial token with index 0 
        # generating a maximum of 100 new tokens
        print(my_tokenizer.decode(model.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0]))

